In [2]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np

current = Path(os.getcwd())
if (current / "data").exists():
    BASE_DIR = current
elif (current.parent / "data").exists():
    BASE_DIR = current.parent
else:
    raise FileNotFoundError("Folder 'data' tidak ditemukan, cek ulang lokasi notebook")

RAW_PATH = BASE_DIR / "data" / "raw" / "Pelayanan Kepuasan - Mahasiswa.xlsx"
OUTPUT_DIR = BASE_DIR / "data" / "clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_PATH   :", RAW_PATH, "| ada?", RAW_PATH.exists())
print("OUTPUT_DIR :", OUTPUT_DIR)

RAW_PATH   : c:\KULIAH\SEM 5\Project Analitika Data\data\raw\Pelayanan Kepuasan - Mahasiswa.xlsx | ada? True
OUTPUT_DIR : c:\KULIAH\SEM 5\Project Analitika Data\data\clean


In [3]:
KOLOM_PII = ["Nomor Gopay"]
KOLOM_PROFIL = ["Timestamp", "Program Studi", "Semester"]
KATA_KUNCI_KLINIK = "klinik"

KOLOM_TEKS = {
    "saran_perbaikan": "Menurut Anda, layanan atau fasilitas apa yang perlu diperbaiki di kampus? Jelaskan perbaikan yang Anda harapkan. (Opsional)"
}

KOLOM_KATEGORI = {
    "prioritas": "Menurut Anda, layanan atau fasilitas mana yang paling perlu ditingkatkan (pilih maksimum 3)?"
}

KOLOM_GATING_KLINIK = "Apakah Anda pernah menggunakan layanan Klinik Kampus PENS?"

SKALA = (1, 4)

In [4]:
df_raw = pd.read_excel(RAW_PATH)
df_raw.columns = [re.sub(r"\s+", " ", str(k)).strip() for k in df_raw.columns]
print("Data mentah:", df_raw.shape)

Data mentah: (418, 74)


In [5]:
df = df_raw.copy()
df.insert(0, "response_id", ["MHS_" + str(i + 1).zfill(4) for i in range(len(df))])

kolom_pii_ada = [k for k in KOLOM_PII if k in df.columns]
df = df.drop(columns=kolom_pii_ada)
print("Kolom PII dibuang:", kolom_pii_ada)

Kolom PII dibuang: ['Nomor Gopay']


In [6]:
semua_kolom = df.columns.tolist()
kolom_likert_semua = [k for k in semua_kolom if "[" in k and "]" in k]

kolom_klinik = [k for k in kolom_likert_semua if KATA_KUNCI_KLINIK.lower() in k.lower()]
kolom_likert_utama = [k for k in kolom_likert_semua if k not in kolom_klinik]

print(f"Likert utama : {len(kolom_likert_utama)} item")
print(f"Likert klinik: {len(kolom_klinik)} item")

Likert utama : 58 item
Likert klinik: 9 item


In [7]:
def ambil_skor(seri):
    hasil = seri.astype(str).str.extract(r"^\s*(\d+)")[0]
    return pd.to_numeric(hasil, errors="coerce")

def cek_konversi(df_asli, df_bersih, kolom, skala, nama):
    sel_asli = df_asli[kolom].notna().sum().sum()
    sel_bersih = df_bersih[kolom].notna().sum().sum()
    di_luar = ((df_bersih[kolom] < skala[0]) | (df_bersih[kolom] > skala[1])).sum().sum()
    print(f"{nama} | terisi asli: {sel_asli} | setelah konversi: {sel_bersih} | di luar skala: {di_luar}")
    if sel_asli != sel_bersih or di_luar > 0:
        print("   PERINGATAN: ada nilai gagal konversi / di luar skala")

df_likert_utama = df[["response_id"] + KOLOM_PROFIL + kolom_likert_utama].copy()
for k in kolom_likert_utama:
    df_likert_utama[k] = ambil_skor(df_likert_utama[k])
cek_konversi(df, df_likert_utama, kolom_likert_utama, SKALA, "Likert Utama")

df_likert_klinik = df[["response_id"] + kolom_klinik].copy()
for k in kolom_klinik:
    df_likert_klinik[k] = ambil_skor(df_likert_klinik[k])
df_likert_klinik = df_likert_klinik.dropna(subset=kolom_klinik, how="all")
cek_konversi(df, df_likert_klinik, kolom_klinik, SKALA, "Likert Klinik")

Likert Utama | terisi asli: 24244 | setelah konversi: 24244 | di luar skala: 0
Likert Klinik | terisi asli: 1035 | setelah konversi: 1035 | di luar skala: 0


In [8]:
def ekstrak_aspek(nama_kolom):
    m = re.search(r"terhadap\s+(?:layanan|fasilitas)?\s*(?:dan\s+layanan\s+)?(.+?)\s*\?", nama_kolom)
    aspek = m.group(1).strip() if m else "TIDAK TERDETEKSI"
    return aspek.replace(" berikut", "").strip()

def ekstrak_pertanyaan(nama_kolom):
    m = re.search(r"\[(.+?)\]", nama_kolom)
    return m.group(1).strip() if m else nama_kolom

def bangun_kamus(kolom_list, prefix):
    kamus = []
    for i, kolom in enumerate(kolom_list, start=1):
        kamus.append({
            "item_kode": f"{prefix}_{str(i).zfill(2)}",
            "pertanyaan": ekstrak_pertanyaan(kolom),
            "aspek": ekstrak_aspek(kolom),
            "nama_kolom_asli": kolom
        })
    return pd.DataFrame(kamus)

df_kamus_utama = bangun_kamus(kolom_likert_utama, "ITEM")
df_kamus_klinik = bangun_kamus(kolom_klinik, "KLINIK")
df_kamus_klinik["aspek"] = "Klinik Kampus"  # samakan label, karena semua item klinik satu aspek

df_kamus = pd.concat([df_kamus_utama, df_kamus_klinik], ignore_index=True)

print(df_kamus["aspek"].value_counts())

# Cek kalau ada yang gagal terdeteksi, perlu diperbaiki manual sebelum lanjut
gagal = df_kamus[df_kamus["aspek"] == "TIDAK TERDETEKSI"]
if len(gagal) > 0:
    print("\nPERINGATAN - item gagal terdeteksi aspeknya:")
    print(gagal[["item_kode", "pertanyaan"]])

aspek
Kemahasiswaan                      10
Administrasi Akademik               9
Administrasi Keuangan               9
Administrasi Umum                   9
Klinik Kampus                       9
Ruang Kuliah                        3
Laboratorium                        3
Perpustakaan                        3
TIK dan Sistem Informasi kampus     3
Toilet kampus                       3
Parkir kampus                       3
Lingkungan Kampus                   3
Name: count, dtype: int64


In [9]:
def hitung_skor_aspek_per_responden(df_likert, df_kamus_subset):
    hasil = pd.DataFrame()
    hasil["response_id"] = df_likert["response_id"]
    for aspek in df_kamus_subset["aspek"].unique():
        kolom_aspek = df_kamus_subset[df_kamus_subset["aspek"] == aspek]["nama_kolom_asli"].tolist()
        kolom_ada = [k for k in kolom_aspek if k in df_likert.columns]
        hasil[aspek] = df_likert[kolom_ada].mean(axis=1).round(2)
    return hasil

df_skor_aspek_utama = hitung_skor_aspek_per_responden(df_likert_utama, df_kamus_utama)
df_skor_aspek_klinik = hitung_skor_aspek_per_responden(df_likert_klinik, df_kamus_klinik)

# Gabung jadi satu matriks responden x aspek (klinik di-join, NaN kalau tidak mengisi)
df_skor_aspek = df_skor_aspek_utama.merge(df_skor_aspek_klinik, on="response_id", how="left")

print("Shape matriks responden x aspek:", df_skor_aspek.shape)
df_skor_aspek.head()

Shape matriks responden x aspek: (418, 13)


,response_id,Administrasi Akademik,Kemahasiswaan,Administrasi Keuangan,Administrasi Umum,Ruang Kuliah,Laboratorium,Perpustakaan,TIK dan Sistem Informasi kampus,Toilet kampus,Parkir kampus,Lingkungan Kampus,Klinik Kampus
0,MHS_0001,4.0,4.0,3.56,3.78,4.0,4.0,4.00,4.00,4.00,4.00,4.00,4.00
1,MHS_0002,3.0,3.0,3.00,3.00,4.0,4.0,3.00,3.00,3.00,2.00,4.00,2.78
2,MHS_0003,4.0,4.0,4.00,4.00,4.0,4.0,4.00,4.00,4.00,4.00,4.00,NaN
3,MHS_0004,3.0,2.9,3.00,3.00,3.0,3.0,2.67,2.67,2.00,3.00,3.33,NaN
4,MHS_0005,3.0,3.0,3.00,3.00,3.0,3.0,3.00,2.67,2.67,2.33,3.00,NaN


In [10]:
def ringkasan_item(df_likert, df_kamus_subset):
    ringkasan = []
    for _, baris in df_kamus_subset.iterrows():
        kolom = baris["nama_kolom_asli"]
        if kolom in df_likert.columns:
            skor = df_likert[kolom]
            ringkasan.append({
                "item_kode": baris["item_kode"],
                "pertanyaan": baris["pertanyaan"],
                "aspek": baris["aspek"],
                "rata_rata": round(skor.mean(), 2),
                "n_responden": int(skor.notna().sum())
            })
    return pd.DataFrame(ringkasan)

df_ringkasan_item_utama = ringkasan_item(df_likert_utama, df_kamus_utama)
df_ringkasan_item_klinik = ringkasan_item(df_likert_klinik, df_kamus_klinik)
df_ringkasan_item = pd.concat([df_ringkasan_item_utama, df_ringkasan_item_klinik], ignore_index=True)

df_ringkasan_aspek = (
    df_ringkasan_item
    .groupby("aspek")
    .agg(rata_rata=("rata_rata", "mean"), jumlah_item=("item_kode", "count"))
    .round(2)
    .reset_index()
    .sort_values("rata_rata")
)

df_ringkasan_item.head()
df_ringkasan_aspek

,aspek,rata_rata,jumlah_item
7,Parkir kampus,2.51,3
10,TIK dan Sistem Informasi kampus,2.72,3
11,Toilet kampus,3.05,3
0,Administrasi Akademik,3.12,9
3,Kemahasiswaan,3.14,10
2,Administrasi Umum,3.18,9
5,Laboratorium,3.19,3
9,Ruang Kuliah,3.20,3
1,Administrasi Keuangan,3.24,9
6,Lingkungan Kampus,3.27,3


In [11]:
df_teks_list = []
for nama, kolom in KOLOM_TEKS.items():
    temp = df[["response_id", kolom]].copy()
    temp.columns = ["response_id", "teks_komentar"]
    temp["jenis_pertanyaan"] = nama
    df_teks_list.append(temp[["response_id", "jenis_pertanyaan", "teks_komentar"]])

df_teks = pd.concat(df_teks_list, ignore_index=True)
df_teks = df_teks.dropna(subset=["teks_komentar"])
df_teks = df_teks[df_teks["teks_komentar"].astype(str).str.strip() != ""]

print(f"Total jawaban teks: {len(df_teks)}")

Total jawaban teks: 314


In [12]:
df_kategori_list = []
for nama, kolom in KOLOM_KATEGORI.items():
    if kolom in df.columns:
        temp = df[["response_id", kolom]].dropna().copy()
        temp["pilihan"] = temp[kolom].astype(str).str.split(",")
        temp = temp.explode("pilihan")
        temp["pilihan"] = temp["pilihan"].str.strip()
        temp["jenis_pertanyaan"] = nama
        df_kategori_list.append(temp[["response_id", "jenis_pertanyaan", "pilihan"]])

df_kategori = pd.concat(df_kategori_list, ignore_index=True)
print(df_kategori["pilihan"].value_counts())

pilihan
Parkir                                 336
Internet/Wi-Fi dan Sistem Informasi    316
Toilet                                  92
Laboratorium                            90
Administrasi Akademik                   69
Ruang Kuliah                            63
Layanan Kemahasiswaan                   60
Lingkungan Kampus                       40
Perpustakaan                            26
Administrasi Umum                       21
Layanan Klinik                          18
Administrasi Keuangan                   12
Name: count, dtype: int64


In [13]:
df_likert_utama.to_csv(OUTPUT_DIR / "likert_mahasiswa_utama_clean.csv", index=False)
df_likert_klinik.to_csv(OUTPUT_DIR / "likert_mahasiswa_klinik_clean.csv", index=False)
df_teks.to_csv(OUTPUT_DIR / "teks_mahasiswa_clean.csv", index=False)
df_kategori.to_csv(OUTPUT_DIR / "kategori_mahasiswa_clean.csv", index=False)

df_kamus.to_csv(OUTPUT_DIR / "kamus_item_aspek_mahasiswa.csv", index=False)
df_skor_aspek.to_csv(OUTPUT_DIR / "skor_aspek_mahasiswa.csv", index=False)
df_ringkasan_item.to_csv(OUTPUT_DIR / "ringkasan_item_mahasiswa.csv", index=False)
df_ringkasan_aspek.to_csv(OUTPUT_DIR / "ringkasan_aspek_mahasiswa.csv", index=False)

print("Selesai. 8 file tersimpan di:", OUTPUT_DIR)

Selesai. 8 file tersimpan di: c:\KULIAH\SEM 5\Project Analitika Data\data\clean
